In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA, IncrementalPCA
from sklearn.preprocessing import StandardScaler
from sklearn import tree
#import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten,Conv2D, MaxPooling2D
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import categorical_focal_crossentropy
from tensorflow.keras.metrics import categorical_focal_crossentropy
from sklearn.model_selection import cross_validate
from sklearn.model_selection import GroupKFold
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pickle
from sklearn.model_selection import GridSearchCV




## Load data

In [5]:
train_data=np.load('Data+Description\\fashion_train.npy')  
train_data_X=train_data[:,:-1]
train_data_y=train_data[:,-1]

In [6]:
with open("pca_model_50_components.pkl", "rb") as file:
    pca_loaded = pickle.load(file)

# Apply the loaded PCA model to new data
#X_new_reduced = pca_loaded.transform(train_data_X)  # X_new is new data
pca_50dim = PCA(n_components=50)
X_new_reduced = pca_50dim.fit_transform(train_data_X)  # Transforms data to 50 components



In [19]:
var_cumu = np.cumsum(pca_loaded.explained_variance_ratio_)*100
print(var_cumu)
k = np.argmax(var_cumu>99)
print("Number of components explaining 95% variance: "+ str(k))

[32.30994384 48.57960044 56.31012675 60.47880909 63.76410878 66.00166641
 67.94284694 69.72410797 71.22079556 72.49360387 73.63000821 74.5998405
 75.4322463  76.17536213 76.90677937 77.58403945 78.25323061 78.8900573
 79.45595373 79.95952339 80.44330693 80.90572714 81.35286908 81.77975969
 82.19907905 82.59517338 82.96879551 83.31153025 83.64223557 83.949844
 84.24883573 84.53881637 84.81803706 85.09251581 85.36176285 85.61501275
 85.86077114 86.09212551 86.31923504 86.53695104 86.74258443 86.94614116
 87.14544106 87.33923074 87.53106054 87.71979325 87.89738887 88.06795005
 88.23295724 88.39017932]
Number of components explaining 95% variance: 0


In [7]:
print(X_new_reduced.shape)
print(train_data_X.shape)


(10000, 50)
(10000, 784)


## Tree

In [4]:
tree_classifier=tree.DecisionTreeClassifier()
scores = cross_validate(tree_classifier, train_data_X, train_data_y, cv=5)

In [5]:
print(scores)
print("test_score_avg - ", sum(scores['test_score'])/5)

{'fit_time': array([10.46029401, 10.84312558, 10.93599653, 11.51850843, 10.95436168]), 'score_time': array([0.03199697, 0.00801563, 0.00699997, 0.00800753, 0.00700474]), 'test_score': array([0.7785, 0.784 , 0.782 , 0.7635, 0.777 ])}
test_score_avg -  0.777


In [6]:
print(scores)
print("test_score_avg - ", sum(scores['test_score'])/5)

{'fit_time': array([10.46029401, 10.84312558, 10.93599653, 11.51850843, 10.95436168]), 'score_time': array([0.03199697, 0.00801563, 0.00699997, 0.00800753, 0.00700474]), 'test_score': array([0.7785, 0.784 , 0.782 , 0.7635, 0.777 ])}
test_score_avg -  0.777


In [7]:
tree_classifier=tree.DecisionTreeClassifier()
scores_pca = cross_validate(tree_classifier, X_new_reduced, train_data_y, cv=5)
print(scores_pca)
print("test_score_avg - ", sum(scores['test_score'])/5)

{'fit_time': array([39.80770564, 35.23415017, 35.00604272, 34.74782467, 38.17797709]), 'score_time': array([0.00700569, 0.00800395, 0.00655675, 0.00798726, 0.00699186]), 'test_score': array([0.7255, 0.74  , 0.7475, 0.7365, 0.729 ])}
test_score_avg -  0.777


## Grid search for trees

In [9]:
param_grid = {
    'criterion': ['gini','entropy'],
    'max_depth': [ 5, 10, 15, 25, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2', None]
}

# Initialize the Decision Tree model
dt = tree.DecisionTreeClassifier()

# Perform grid search with 5-fold cross-validation
grid_search = GridSearchCV(estimator=dt, param_grid=param_grid, cv=5, scoring='accuracy')
grid_search.fit(X_new_reduced, train_data_y)

# Best parameters and score
print("Best parameters:", grid_search.best_params_)
print("Best cross-validated accuracy score:", grid_search.best_score_)

KeyboardInterrupt: 

## FFN

In [1]:
"""def create_model(input_dim):
    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))  # Output layer for binary classification
    model.compile(optimizer=Adam(learning_rate=0.001), loss='binary_crossentropy', metrics=['accuracy'])
    return model"""

def create_multiclass_model(input_dim, num_classes):
    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu'))  # First hidden layer
    model.add(Dense(64, activation='relu'))  # Second hidden layer
    model.add(Dense(32, activation='relu'))  # Third hidden layer
    model.add(Dense(num_classes, activation='softmax'))  # Output layer for multi-class classification
    
    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',  # or 'categorical_crossentropy' if using one-hot encoding
                  metrics=['accuracy'])
    return model

In [2]:

# Set up 5-Fold Cross-Validation
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []

for fold, (train_index, val_index) in enumerate(kf.split(train_data_X)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = train_data_X[train_index], train_data_X[val_index]
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    
    # Create a new instance of the model
    model = create_multiclass_model(input_dim=train_data_X.shape[1], num_classes=len(np.unique(train_data_y)))
    
    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=1)  # Get the predicted class for each sample

    # Calculate accuracy for the current fold
    accuracy = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy)

    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")

NameError: name 'KFold' is not defined

In [10]:
# pca
kf = KFold(n_splits=5, shuffle=True)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []

for fold, (train_index, val_index) in enumerate(kf.split(X_new_reduced)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = X_new_reduced[train_index], X_new_reduced[val_index]
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    print(X_train.shape)
    # Create a new instance of the model
    model = create_multiclass_model(input_dim=X_new_reduced.shape[1], num_classes=len(np.unique(train_data_y)))
    
    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=1)  # Get the predicted class for each sample

    # Calculate accuracy for the current fold
    accuracy_pca = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy_pca)

    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy_pca:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy_pca = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy_pca:.4f}")

Training fold 1
(8000, 50)


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 1 Accuracy: 0.8170
Classification Report for Fold 1:
               precision    recall  f1-score   support

           0       0.78      0.78      0.78       395
           1       0.97      0.96      0.96       379
           2       0.88      0.78      0.83       420
           3       0.93      0.83      0.88       414
           4       0.59      0.74      0.66       392

    accuracy                           0.82      2000
   macro avg       0.83      0.82      0.82      2000
weighted avg       0.83      0.82      0.82      2000

Confusion Matrix for Fold 1:
 [[309   2   5   6  73]
 [  3 365   2   4   5]
 [  8   1 326   1  84]
 [ 20   9   4 345  36]
 [ 55   1  33  14 289]]
Training fold 2
(8000, 50)


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 2 Accuracy: 0.8310
Classification Report for Fold 2:
               precision    recall  f1-score   support

           0       0.86      0.75      0.80       385
           1       0.98      0.95      0.97       406
           2       0.80      0.89      0.84       400
           3       0.81      0.94      0.87       406
           4       0.71      0.62      0.66       403

    accuracy                           0.83      2000
   macro avg       0.83      0.83      0.83      2000
weighted avg       0.83      0.83      0.83      2000

Confusion Matrix for Fold 2:
 [[288   0  10  34  53]
 [  1 387   2  13   3]
 [  2   1 354  10  33]
 [  4   4   3 382  13]
 [ 41   1  76  34 251]]
Training fold 3
(8000, 50)


...

63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 3 Accuracy: 0.8110
Classification Report for Fold 3:
               precision    recall  f1-score   support

           0       0.78      0.75      0.77       397
           1       0.98      0.94      0.96       381
           2       0.76      0.90      0.83       402
           3       0.83      0.89      0.86       389
           4       0.70      0.60      0.65       431

    accuracy                           0.81      2000
   macro avg       0.81      0.82      0.81      2000
weighted avg       0.81      0.81      0.81      2000

Confusion Matrix for Fold 3:
 [[299   1  11  27  59]
 [  0 357   5  17   2]
 [  4   0 361   5  32]
 [ 14   5   7 348  15]
 [ 64   1  89  20 257]]
Training fold 4
(8000, 50)


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 4 Accuracy: 0.8205
Classification Report for Fold 4:
               precision    recall  f1-score   support

           0       0.79      0.75      0.77       432
           1       0.96      0.98      0.97       369
           2       0.83      0.87      0.85       406
           3       0.89      0.83      0.86       397
           4       0.66      0.70      0.68       396

    accuracy                           0.82      2000
   macro avg       0.82      0.82      0.82      2000
weighted avg       0.82      0.82      0.82      2000

Confusion Matrix for Fold 4:
 [[322   1  12  17  80]
 [  0 360   1   7   1]
 [ 12   2 353   6  33]
 [ 20   7   9 330  31]
 [ 53   6  50  11 276]]
Training fold 5
(8000, 50)


c:\Users\user\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


63/63 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
Fold 5 Accuracy: 0.7950
Classification Report for Fold 5:
               precision    recall  f1-score   support

           0       0.79      0.74      0.77       424
           1       0.96      0.97      0.96       412
           2       0.70      0.89      0.78       373
           3       0.87      0.84      0.86       399
           4       0.64      0.54      0.58       392

    accuracy                           0.80      2000
   macro avg       0.79      0.79      0.79      2000
weighted avg       0.80      0.80      0.79      2000

Confusion Matrix for Fold 5:
 [[315   6  19  18  66]
 [  1 399   2   9   1]
 [  7   0 331   5  30]
 [ 17   7  17 334  24]
 [ 57   4 104  16 211]]

Average Accuracy across 5 folds: 0.8149


In [ ]:
fnn_classifier = create_multiclass_model()

In [ ]:
scores

## CNN

In [11]:

def create_cnn_model(input_shape, num_classes):
    model = Sequential()
    model.add(Conv2D(32, (3, 3), activation='relu', input_shape=input_shape))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Conv2D(64, (3, 3), activation='relu'))
    model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dense(256, activation='relu'))
    model.add(Dense(128, activation='relu'))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  # Softmax for multi-class classification
    
    # Compile the model
    model.compile(optimizer=Adam(learning_rate=0.001), 
                  loss=categorical_focal_crossentropy, 
                  metrics=['accuracy'])
    return model

In [1]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []

for fold, (train_index, val_index) in enumerate(kf.split(train_data_X)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = train_data_X[train_index].reshape(-1, 28, 28,1), train_data_X[val_index].reshape(-1, 28, 28,1)
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    #print(X_train[0])
    
    # Create a new instance of the CNN model
    model = create_cnn_model(input_shape=train_data_X.reshape(-1, 28, 28, 1).shape[1:], num_classes=len(np.unique(train_data_y)))
    
    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=1)  # Convert probabilities to class predictions
    
    # Calculate accuracy for the current fold
    accuracy = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy)

    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")

NameError: name 'KFold' is not defined

In [24]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
fold_accuracies = []
all_classification_reports = []
all_confusion_matrices = []

#ipca = IncrementalPCA(n_components=50)
#X_new_reduced_CNN=(X_new_reduced).reshape(-1, 5, 10, 1)

for fold, (train_index, val_index) in enumerate(kf.split(X_new_reduced)):
    print(f"Training fold {fold + 1}")
    
    # Split the data into training and validation sets for the current fold
    X_train, X_val = X_new_reduced[train_index].reshape(-1, 5, 10, 1), X_new_reduced[val_index].reshape(-1, 5, 10, 1)
    y_train, y_val = train_data_y[train_index], train_data_y[val_index]
    #print(X_train[0])
    
    # Create a new instance of the CNN model
    model = create_cnn_model(input_shape=X_new_reduced.reshape(-1, 5, 10, 1).shape[1:], num_classes=len(np.unique(train_data_y)))
    
    # Train the model
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)
    
    # Evaluate the model on the validation set
    y_val_pred = np.argmax(model.predict(X_val), axis=1)  # Convert probabilities to class predictions
    
    # Calculate accuracy for the current fold
    accuracy = accuracy_score(y_val, y_val_pred)
    fold_accuracies.append(accuracy)

    # Generate classification report and confusion matrix for detailed metrics
    class_report = classification_report(y_val, y_val_pred, output_dict=True)
    conf_matrix = confusion_matrix(y_val, y_val_pred)
    
    # Store the classification report and confusion matrix for the fold
    all_classification_reports.append(class_report)
    all_confusion_matrices.append(conf_matrix)

    print(f"Fold {fold + 1} Accuracy: {accuracy:.4f}")
    print(f"Classification Report for Fold {fold + 1}:\n", classification_report(y_val, y_val_pred))
    print(f"Confusion Matrix for Fold {fold + 1}:\n", conf_matrix)

# Calculate the average accuracy across all folds
average_accuracy = np.mean(fold_accuracies)
print(f"\nAverage Accuracy across 5 folds: {average_accuracy:.4f}")

Training fold 1


ValueError: Computed output size would be negative. Received `inputs shape=(None, 1, 4, 32)`, `kernel shape=(3, 3, 32, 64)`, `dilation_rate=[1 1]`.